# Genex Interview + Activity Brain

This notebook implements the current **Genex developmental interview and activity-planning brain** for children, currently focused on roughly **ages 0–5** using the **CDC milestone tables** as the main structured developmental reference. The goal of this file is to convert parent-reported information into a practical, non-diagnostic developmental support plan that can later be integrated into the Genex app. The system is designed to be transparent, easy to inspect, and simple to debug while the underlying logic is still being refined. The current architecture follows a staged pipeline: **intake → delay estimation → milestone interview by category → developmental age estimation → activity-bank generation → weekly schedule generation**. This reflects the design direction of separating **category-level activity generation** from **weekly schedule construction**, rather than trying to create a full day-by-day plan in one step. 

## What this notebook is for

This notebook is the working prototype of the Genex “brain.” It is intended to:
- collect a child profile from parent input
- estimate which developmental categories are likely affected
- ask milestone-based questions from the CDC tables
- estimate a rough developmental age by category
- identify which categories likely need support
- generate a small bank of home activities for each category
- build a weekly plan that respects the family’s available daily time

The notebook is **not** intended to diagnose a child or replace clinician judgment. It is a structured support tool for parent guidance, advisor review, and iterative development toward a production Genex system. The broader Genex direction is consistent with your larger project framing: structured intake, milestone interviewing, grounded evidence, and downstream activity personalization. 

## Inputs

The notebook currently expects parent-provided intake information such as:
- child name
- chronological age in **months**
- diagnosis / condition, if known
- main parent concern
- available daily time for practice / play

After intake, the notebook asks milestone questions by developmental category. The questions are selected from the CDC milestone table and grouped around an estimated developmental range. The parent answers each question using a constrained answer format such as:
- `yes`
- `sometimes`
- `with_help`
- `no`
- `not_sure`

These answers are then used to estimate a developmental level for each category in a conservative way. 

## Developmental categories

The current notebook organizes the developmental interview into four major categories:
- **Movement / Physical**
- **Social / Emotional**
- **Language / Communication**
- **Cognitive / Adaptive**

Each category is processed separately so the system can produce a more granular developmental profile instead of a single overall score. This category-based structure is also what allows the later activity planning to emphasize some domains more than others. 

## High-level process

### 1. Intake
The notebook first collects a basic child profile and available daily time. This information becomes the shared state used by the rest of the pipeline.

### 2. Delay Estimator Agent
Before asking milestone questions, the notebook estimates a **starting delay in months** for each category. This is not a diagnosis and not a final developmental age. It is only a rough anchor used to decide which milestone age bands should be asked first.

### 3. Milestone Q&A Agent
For each category, the notebook retrieves milestone questions from the CDC table around the estimated developmental level and asks the parent those questions directly. The notebook prints the question/answer transcript and a month-band summary so the logic can be reviewed and debugged.

### 4. Developmental Age Estimation
Using the parent’s answers, the notebook estimates a rough developmental age for each category. The current design is intentionally conservative: a category should not be considered “confirmed” at a higher level unless there is enough evidence from the answers at that band.

### 5. Category Activity Bank Generation
Instead of creating a full weekly plan directly, the notebook first generates a **category-specific bank of activities**. Each activity is meant to be practical for home and includes things like:
- title
- instructions
- estimated duration
- materials
- level / difficulty

### 6. Weekly Schedule Builder
The notebook then uses:
- daily available time
- milestone gap by category
- activity banks

to produce a weekly schedule. The schedule is meant to cover all categories that need support while giving more time to categories with larger milestone gaps. This follows the design principle that categories with bigger developmental gaps should receive greater weekly emphasis. 

## Outputs

The notebook currently produces several outputs:

### A. Printed debug output
During execution, it prints:
- intake banner
- delay estimates by category
- milestone questions asked
- parent answers
- month-band summaries
- estimated developmental age by category

This makes it easier to inspect whether the logic is behaving sensibly for a known case. 

### B. Structured state object
Internally, the notebook stores a structured state object that contains:
- child profile
- delay estimates
- Q&A responses
- estimated developmental ages
- activity banks
- weekly slot allocation
- weekly schedule

This state is intended to become the interface between the current notebook prototype and a future deployed Genex backend.

### C. Summary dataframe
At the end, the notebook returns a summary table that typically includes:
- child information
- category
- estimated delay
- estimated developmental age
- milestone gap
- whether special support is needed
- activity-bank / scheduling information

### D. Weekly schedule
The final output is a practical weekly schedule that selects activities from the activity banks and distributes them across the week according to daily time limits and relative category need. 

## Design principles

This notebook is built around a few important design choices:

1. **Non-diagnostic framing**  
   The system is meant to support parents and organize developmental information, not diagnose.

2. **Structured milestone logic first**  
   The CDC milestone tables are currently the primary developmental backbone for the 0–5 age range. Your broader Genex plan may later extend older ages using Bright Futures, but this notebook currently centers on the CDC logic first. 

3. **Category-specific planning**  
   Each domain is handled separately so the activity plan can be more personalized.

4. **Activity-bank then scheduler architecture**  
   The notebook intentionally separates “what activities exist for this category” from “how should activities be distributed across the week.” This makes the system easier to extend, debug, and deploy later. 

5. **Transparency over elegance**  
   At this stage, the notebook is designed to be readable and easy to inspect, even if some parts will later be refactored into cleaner modules or APIs.

## Current limitations

This notebook is still a prototype and has important limitations:
- developmental estimation is still heuristic and needs more advisor validation
- age coverage is currently centered on the CDC milestone range and not yet fully extended to adolescence
- activity quality still depends on prompt quality and downstream review
- some edge cases may require additional category-specific rules
- the output should always be interpreted as support guidance to discuss with clinicians, not as a medical conclusion


In [3]:
# ---------------------------
# Imports & Setups 
# ----------------------------

import os
import re
import json
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    
load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT/"outputs"/"QuestiontoActivity"
OUTPUT_DIR.mkdir(parents = True, exist_ok = True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

if OpenAI is None: 
    print("Warning: openai package is not available yet")
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: Openai API key is not visible. LLM backed steps will fail until the key is set")

In [4]:
# --------------------------------
# Initializations 
# ---------------------------------

DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotional": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",

    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",   # <-- important fix
    "social_emotional": "social_and_emotional",
    "social": "social_and_emotional",

    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",

    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.4,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}


VALID_ANSWERS = set(ANSWER_SCORES.keys())

In [5]:
def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or notebooks parent folder."
    )

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [f"{row.category_key}_{row.months}_{i}" for i, row in enumerate(df.itertuples(), start=1)]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())
print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows:", len(cdc_df))

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

def normalize_answer(answer_text: str) -> str:
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

Loaded CDC file: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


In [6]:
# ----------------------------------
# State + intake
# ----------------------------------

def new_state() -> Dict[str, Any]:
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "support_recommendations": {},
        "activity_banks": {},
        "weekly_slot_allocation": {},
        "weekly_schedule": {},
    }

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    print_banner("INTAKE AGENT")

    name = input("What is your child's name? ").strip()
    chronological_months = int(
        input("How old is your child in months? (for example: 18, 24, 36, 48, 60) ").strip()
    )
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()
    daily_time_min = int(
        input("About how many minutes per day can you usually spend helping or playing with your child? ").strip()
    )

    state["child"] = {
        "name": name,
        "chronological_months": chronological_months,
        "diagnosis": diagnosis,
        "concern": concern,
        "daily_time_min": daily_time_min,
    }

    return state["child"]

In [7]:
# ---------------------------------------
# Delay Estimator 
# ---------------------------------------
def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG[category_key]["display"]

    diagnosis_l = (diagnosis or "").lower()
    concern_l = (concern or "").lower()

    fallback_by_category = {
        "movement_and_physical": 3,
        "language_and_communication": 3,
        "social_and_emotial": 6,
        "cognitive": 6,
    }

    domain_keywords = {
        "movement_and_physical": [
            "motor", "movement", "walk", "run", "jump", "balance", "coordination",
            "fine motor", "gross motor", "grasp", "hand", "writing"
        ],
        "language_and_communication": [
            "speech", "language", "talk", "communication", "words", "sentence",
            "understand", "expressive", "receptive", "verbal"
        ],
        "social_and_emotial": [
            "social", "peer", "friend", "play", "emotion", "emotional",
            "behavior", "anger", "meltdown", "interaction", "turn taking",
            "regulation"
        ],
        "cognitive": [
            "attention", "focus", "concentration", "school", "learning", "routine",
            "executive", "task", "independent", "adaptive", "toilet", "dressing"
        ],
    }

    has_domain_signal = any(
        kw in concern_l for kw in domain_keywords.get(category_key, [])
    )

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    client = OpenAI()

    prompt = f"""
You are a pediatric developmental delay estimator agent for children ages 0 to 5 years.

Your job is to estimate a SINGLE STARTING DELAY in months for one developmental domain only.

This is NOT a diagnosis.
This is NOT a final developmental age.
This is ONLY a rough starting anchor for question selection.

Definition:
delay_months = chronological age in months - estimated functional developmental age in this specific domain

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Instructions:
1. Think only about THIS domain, not overall development.
2. If the diagnosis or concern does NOT meaningfully affect this domain, return 0 to 6 months.
3. If this domain IS affected, estimate the child's functional developmental level in this domain, then convert it into delay_months.
4. Be conservative but realistic.
5. Never exceed the child's chronological age.
6. Return only one integer number of months.
7. Return strict JSON only.

Examples:

Example 1:
Child is 60 months old with Down syndrome.
Domain = Language / Communication.
If you think language skills are functioning around 24 months,
return:
{{"delay_months": 36, "reason": "Language development is likely meaningfully affected in this child."}}

Example 2:
Child is 48 months old with autism.
Domain = Social / Emotional.
If you think social interaction skills are functioning around 24 months,
return:
{{"delay_months": 24, "reason": "Social-emotional development is likely meaningfully affected in this child."}}

Example 3:
Child is 60 months old with cerebral palsy.
Domain = Movement / Physical.
If you think motor skills are functioning around 30 months,
return:
{{"delay_months": 30, "reason": "Movement and physical development are likely meaningfully affected in this child."}}

Example 4:
Child is 30 months old with speech delay.
Domain = Language / Communication.
If you think language skills are functioning around 18 months,
return:
{{"delay_months": 12, "reason": "Language development is likely delayed based on the concern."}}

Example 5:
Child is 36 months old with global developmental delay.
Domain = Cognitive / Adaptive.
If you think adaptive / cognitive functioning is around 18 months,
return:
{{"delay_months": 18, "reason": "Cognitive and adaptive development are likely meaningfully affected."}}

Example 6:
Child is 48 months old with autism.
Domain = Movement / Physical.
If there is no meaningful motor concern,
return:
{{"delay_months": 0, "reason": "There is little reason to think this domain is significantly affected."}}

Example 7:
Child is 60 months old with Down syndrome.
Domain = Social / Emotional.
If you think social skills are only mildly affected,
return:
{{"delay_months": 6, "reason": "This domain may be mildly affected but not severely delayed."}}

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )

        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(parsed.get("delay_months", fallback_by_category.get(category_key, 6)))
        delay_months = max(0, min(delay_months, chronological_months))

        # Guardrail: if the concern does not really point to this domain,
        # do not allow a large delay estimate.
        if not has_domain_signal and delay_months > 6:
            delay_months = 6

        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }

    except Exception as e:
        return {
            "delay_months": fallback_by_category.get(category_key, 6),
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }
    

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]

In [8]:
# -----------------------------------------
# Milestone Questionnaire 
# ------------------------------------------
def build_milestone_questions(
    state: Dict[str, Any],
    category_key: str,
    window_months: int = 24,
    target_questions_per_band: int = 4,
    max_bands: int = 3,
    max_questions_total: int = 12,
) -> List[Dict[str, Any]]:
    child = state["child"]
    chrono_months = min(child["chronological_months"], 60)
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    # Available month bands in the chosen window
    available_months = sorted(subset["months"].unique().tolist())

    # Ask nearest bands first
    ordered_months = sorted(
        available_months,
        key=lambda m: (abs(m - approx_dev_months), m)
    )[:max_bands]

    questions = []
    for month in ordered_months:
        month_rows = subset[subset["months"] == month].sort_values("milestone")
        month_rows = month_rows.head(target_questions_per_band)

        for _, row in month_rows.iterrows():
            questions.append({
                "question_id": row["question_id"],
                "months": int(row["months"]),
                "milestone": row["milestone"],
                "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
            })

    return questions[:max_questions_total]

In [9]:
# ------------------------------------------
# Compute developmental age from the answers
# -------------------------------------------

def compute_dev_age_from_answers(
    answers: List[Dict[str, Any]],
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> int:
    """
    Conservative band-based heuristic.

    A month band is confirmed only if:
    - it has at least `min_yes_confirm` YES answers
    - and YES answers are at least `yes_ratio_confirm` of the answers in that band

    Rules:
    - developmental age = highest confirmed band
    - if no confirmed band, use one lower band than the earliest partial band
    - if only NO answers, use the minimum asked band
    """
    if not answers:
        return 6

    # Group by month band
    band_summary = {}
    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
            }

        band_summary[month]["total"] += 1

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    answered_months = sorted(band_summary.keys())

    confirmed_months = []
    partial_months = []

    for month in answered_months:
        total = band_summary[month]["total"]
        yes_count = band_summary[month]["yes"]
        partial_count = band_summary[month]["partial"]

        yes_ratio = yes_count / total if total > 0 else 0

        if yes_count >= min_yes_confirm and yes_ratio >= yes_ratio_confirm:
            confirmed_months.append(month)
        elif yes_count > 0 or partial_count > 0:
            partial_months.append(month)

    if confirmed_months:
        return int(max(confirmed_months))

    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)

    return int(min(answered_months))

In [10]:
# -------------------------------------
# Summarize answers by band 
#--------------------------------------

def summarize_answers_by_band(answers: List[Dict[str, Any]]) -> Dict[int, Dict[str, Any]]:
    band_summary = {}

    for a in answers:
        month = int(a["months"])
        norm = a["norm_answer"]

        if month not in band_summary:
            band_summary[month] = {
                "total": 0,
                "yes": 0,
                "partial": 0,
                "no": 0,
                "items": [],
            }

        band_summary[month]["total"] += 1
        band_summary[month]["items"].append({
            "milestone": a["milestone"],
            "answer": norm,
        })

        if norm == "yes":
            band_summary[month]["yes"] += 1
        elif norm in {"sometimes", "with_help", "not_sure"}:
            band_summary[month]["partial"] += 1
        else:
            band_summary[month]["no"] += 1

    for month in band_summary:
        total = band_summary[month]["total"]
        yes = band_summary[month]["yes"]
        band_summary[month]["yes_ratio"] = round(yes / total, 2) if total else 0.0

    return dict(sorted(band_summary.items()))

In [11]:
# ---------------------------------------
# Question and answer - live 
# ---------------------------------------
def qna_agent_live(
    state: Dict[str, Any],
    category_key: str,
    min_yes_confirm: int = 2,
    yes_ratio_confirm: float = 0.70,
) -> Dict[str, Any]:
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")

    if "qna" not in state:
        state["qna"] = {}
    if "dev_age" not in state:
        state["dev_age"] = {}

    questions = build_milestone_questions(
        state,
        category_key,
        window_months=24,
        target_questions_per_band=4,
        max_bands=3,
        max_questions_total=12,
    )

    if not questions:
        raise ValueError(
            f"No milestone questions found for category_key={category_key}. "
            f"Check category mapping and CDC normalization."
        )

    asked = []

    questions_by_month = {}
    for q in questions:
        questions_by_month.setdefault(q["months"], []).append(q)

    for month in sorted(questions_by_month.keys()):
        print(f"\n--- Asking {DOMAIN_CONFIG[category_key]['display']} milestones around {month} months ---")
        for q in questions_by_month[month]:
            ans = input(q["question_text"] + "\n").strip()
            norm = normalize_answer(ans)
            asked.append({
                "question_id": q["question_id"],
                "months": q["months"],
                "milestone": q["milestone"],
                "raw_answer": ans,
                "norm_answer": norm,
                "score": ANSWER_SCORES[norm],
            })

    dev_age = compute_dev_age_from_answers(
        asked,
        min_yes_confirm=min_yes_confirm,
        yes_ratio_confirm=yes_ratio_confirm,
    )

    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]

    print("\nQuestion / answer transcript:")
    for item in asked:
        print(f"- [{item['months']}m] {item['milestone']} -> {item['norm_answer']}")

    band_summary = summarize_answers_by_band(asked)
    print("\nBand summary:")
    for month, info in band_summary.items():
        print(
            f"  {month}m | total={info['total']} | yes={info['yes']} | "
            f"partial={info['partial']} | no={info['no']} | yes_ratio={info['yes_ratio']}"
        )

    print(
        f"\nEstimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: "
        f"{dev_age} months (chronological age {chrono} months)"
    )

    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
        "band_summary": band_summary,
    }

In [12]:
# ----------------------------------------
# When there is no special support needed
# ----------------------------------------
def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)

    # Only suppress support when milestone delay is truly minimal
    return (chrono - dev_age) <= 4 and delay_est <= 6

In [13]:
# -------------------------------------
# Select next milestone 
# -------------------------------------
def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)

    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    # Start at the estimated developmental age itself so the plan can include
    # current-level practice plus near next-step skills.
    # We can revise this later if we find the system is still too optimistic
    # or the child needs more reinforcement below the estimated level.
    min_m = max(2, dev_age)
    max_m = min(60, dev_age + 12)

    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)

    if subset.empty:
        return {
            "status": "success",
            "milestones": [],
        }

    subset = subset.sort_values(["months", "milestone"]).copy()

    milestones = []
    seen = set()

    for _, row in subset.iterrows():
        key = (int(row["months"]), str(row["milestone"]).strip())
        if key in seen:
            continue
        seen.add(key)

        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
    }

In [14]:
#---------------------------------------
# generate_category_activity_bank
#---------------------------------------
def generate_category_activity_bank(
    state: Dict[str, Any],
    category_key: str,
    model: str = "gpt-4o-mini",
    activities_per_category: int = 6,
) -> Dict[str, Any]:
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]

    next_steps = select_next_milestones(state, category_key)

    if next_steps["status"] == "no_special_support":
        state["activity_banks"][category_key] = {
            "status": "no_special_support",
            "summary": next_steps["message"],
            "activities": [],
        }
        return state["activity_banks"][category_key]

    dev_age = state["dev_age"][category_key]
    chrono_months = min(child["chronological_months"], 60)
    milestone_gap = max(0, chrono_months - dev_age)

    milestone_lines = "\n".join(
        [f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]]
    )
    if not milestone_lines:
        milestone_lines = "- No specific milestone items available in this range."

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        # If milestone list is short, pad with generic items
        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback activity bank used for {category_display}.",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

    client = OpenAI()

    prompt = f"""
You are a pediatric home-support planning agent helping a parent at home.

This is NOT a diagnosis and NOT a formal treatment plan.
Create a CATEGORY ACTIVITY BANK, not a day-by-day schedule.

Child:
- Name: {child['name']}
- Chronological age: {child['chronological_months']} months
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Estimated developmental age in this category: {dev_age} months
- Estimated milestone gap in this category: {milestone_gap} months

Relevant milestone targets:
{milestone_lines}

Task:
Create {activities_per_category} realistic home activities for this category.

Instructions:
1. Activities must fit the child's chronological age and estimated developmental level.
2. Activities should be practical for home.
3. Keep language parent-friendly.
4. Include a mix of current-level practice and near next-step practice.
5. Most activities should be short and realistic for home: usually 3, 5, 7, or 10 minutes.
6. Avoid making many 15-minute activities unless clearly justified.
7. Rare exception: some activities may naturally benefit from longer continuous time, such as a short playdate, playground peer practice, or group social activity.
8. Only mark an activity as extended if it is clearly justified.
9. Extended activities should usually be 15 or 20 minutes, not longer than 20 unless parents have more than 20 minutes available per day.
10. Do not make more than 1 or 2 extended activities in this category.
11. Return strict JSON only.

Required JSON:
{{
  "summary": "...",
  "activities": [
    {{
      "activity_id": "1",
      "title": "...",
      "instructions": "...",
      "duration_min": 5,
      "materials": "...",
      "level": "current_or_next",
      "category": "<category name>",
      "is_extended_activity": false,
      "extended_reason": ""
    }}
  ]
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": "You return strict JSON only and stay non-diagnostic, practical, and parent-friendly."
                },
                {"role": "user", "content": prompt},
            ],
        )

        bank = json.loads(resp.choices[0].message.content)

        # safety cleanup
        if "activities" not in bank or not isinstance(bank["activities"], list):
            bank["activities"] = []

        for idx, activity in enumerate(bank["activities"], start=1):
            activity.setdefault("activity_id", f"{category_key}_{idx}")
            activity.setdefault("category", category_display)
            activity.setdefault("duration_min", 5)
            activity.setdefault("materials", "common household items")
            activity.setdefault("level", "current_or_next")
            activity.setdefault("is_extended_activity", False)
            activity.setdefault("extended_reason", "")

        state["activity_banks"][category_key] = {
            "status": "success",
            "summary": bank.get("summary", f"Created activity bank for {category_display}."),
            "activities": bank["activities"][:activities_per_category],
        }
        return state["activity_banks"][category_key]

    except Exception as e:
        fallback_activities = []
        for i, m in enumerate(next_steps["milestones"][:activities_per_category], start=1):
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"Practice: {m['milestone']}",
                "instructions": f"Use a short, play-based home activity to practice: {m['milestone']}.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        while len(fallback_activities) < activities_per_category:
            i = len(fallback_activities) + 1
            fallback_activities.append({
                "activity_id": f"{category_key}_{i}",
                "title": f"{category_display} practice {i}",
                "instructions": f"Do one short parent-guided home activity that supports {category_display.lower()} skills.",
                "duration_min": 5,
                "materials": "common toys or household items",
                "level": "current_or_next",
                "category": category_display,
                "is_extended_activity": False,
                "extended_reason": "",
            })

        state["activity_banks"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback activity bank used for {category_display} because OpenAI failed: {e}",
            "activities": fallback_activities,
        }
        return state["activity_banks"][category_key]

In [15]:
#---------------------------------------
# allocate_weekly_slots
#---------------------------------------
def allocate_weekly_slots(state: Dict[str, Any]) -> Dict[str, Any]:
    child = state["child"]

    if "daily_time_min" not in state["child"]:
        raise ValueError(
            "Missing daily_time_min in state['child']. "
            "Check intake_agent_live() and make sure it stores daily_time_min."
        )

    chrono = min(child["chronological_months"], 60)
    daily_time_min = int(child["daily_time_min"])
    weekly_minutes = daily_time_min * 5

    supported_categories = []
    gap_by_category = {}

    for category_key in DOMAIN_CONFIG:
        if not no_special_support_needed(state, category_key):
            supported_categories.append(category_key)
            dev_age = state["dev_age"][category_key]
            gap_by_category[category_key] = max(0, chrono - dev_age)

    if not supported_categories:
        allocation = {
            "daily_time_min": daily_time_min,
            "weekly_minutes": weekly_minutes,
            "supported_categories": [],
            "gap_by_category": {},
            "target_minutes_by_category": {},
        }
        state["weekly_slot_allocation"] = allocation
        return allocation

    # Give each supported category at least a baseline share
    base_minutes_per_category = max(5, daily_time_min // 2)

    target_minutes_by_category = {
        k: base_minutes_per_category for k in supported_categories
    }

    used_minutes = base_minutes_per_category * len(supported_categories)
    remaining_minutes = max(0, weekly_minutes - used_minutes)

    weights = {k: max(1, gap_by_category[k]) for k in supported_categories}
    total_weight = sum(weights.values())

    if total_weight > 0 and remaining_minutes > 0:
        for k in supported_categories:
            extra = round(remaining_minutes * (weights[k] / total_weight))
            target_minutes_by_category[k] += extra

    # Normalize if rounding overshoots
    total_target = sum(target_minutes_by_category.values())
    while total_target > weekly_minutes:
        biggest = max(target_minutes_by_category, key=target_minutes_by_category.get)
        if target_minutes_by_category[biggest] > 5:
            target_minutes_by_category[biggest] -= 1
            total_target -= 1
        else:
            break

    allocation = {
        "daily_time_min": daily_time_min,
        "weekly_minutes": weekly_minutes,
        "supported_categories": supported_categories,
        "gap_by_category": gap_by_category,
        "target_minutes_by_category": target_minutes_by_category,
    }

    state["weekly_slot_allocation"] = allocation
    return allocation

In [16]:
#------------------------------------------------
# pick_activity_that_fits the time parent have 
#------------------------------------------------
def pick_activity_that_fits(
    activities: List[Dict[str, Any]],
    used_indices: set,
    remaining_minutes: int,
) -> Optional[Dict[str, Any]]:
    """
    Pick the shortest unused activity that fits in the remaining daily time.
    If nothing fits, return None.
    """
    candidates = []

    for idx, activity in enumerate(activities):
        if idx in used_indices:
            continue
        duration = int(activity.get("duration_min", 5))
        if duration <= remaining_minutes:
            candidates.append((duration, idx, activity))

    if not candidates:
        return None

    # shortest fitting activity first
    candidates = sorted(candidates, key=lambda x: x[0])
    duration, idx, activity = candidates[0]
    used_indices.add(idx)
    return activity

In [17]:
#---------------------------------------
# pick_weekly_bonus_extended_activity
#---------------------------------------
def pick_weekly_bonus_extended_activity(
    state: Dict[str, Any],
    extended_in_schedule_threshold: int = 15,
    bonus_extended_cap_min: int = 20,
) -> Optional[Dict[str, Any]]:
    """
    If daily time is small, allow one optional weekly extended activity
    outside the normal weekday daily-minute budget.
    """

    daily_time_min = int(state["child"]["daily_time_min"])

    # If the family already has enough daily time, no need for a separate bonus rule.
    if daily_time_min >= extended_in_schedule_threshold:
        return None

    allocation = state.get("weekly_slot_allocation", {})
    gap_by_category = allocation.get("gap_by_category", {})

    # Prefer categories with bigger gaps; social/language often benefit from longer natural activities.
    candidate_categories = sorted(
        gap_by_category.keys(),
        key=lambda k: gap_by_category[k],
        reverse=True,
    )

    for category_key in candidate_categories:
        bank = state.get("activity_banks", {}).get(category_key, {})
        activities = bank.get("activities", [])

        for activity in activities:
            if not activity.get("is_extended_activity", False):
                continue

            duration = int(activity.get("duration_min", 5))
            if duration > bonus_extended_cap_min:
                continue

            return {
                "category_key": category_key,
                "category": DOMAIN_CONFIG[category_key]["display"],
                "title": activity.get("title"),
                "instructions": activity.get("instructions"),
                "duration_min": duration,
                "materials": activity.get("materials", "common household items"),
                "level": activity.get("level", "current_or_next"),
                "extended_reason": activity.get("extended_reason", ""),
            }

    return None

In [18]:
#---------------------------------------
# build_weekly_schedule
#---------------------------------------
def build_weekly_schedule(state: Dict[str, Any]) -> Dict[str, Any]:
    if "weekly_slot_allocation" not in state:
        allocate_weekly_slots(state)

    allocation = state["weekly_slot_allocation"]
    daily_time_min = allocation["daily_time_min"]
    target_minutes_by_category = allocation["target_minutes_by_category"]

    days = {
        "day_1": {"items": [], "total_minutes": 0},
        "day_2": {"items": [], "total_minutes": 0},
        "day_3": {"items": [], "total_minutes": 0},
        "day_4": {"items": [], "total_minutes": 0},
        "day_5": {"items": [], "total_minutes": 0},
    }

    if not target_minutes_by_category:
        state["weekly_schedule"] = {
            "status": "no_special_support",
            "summary": "No categories need a scheduled weekly activity plan right now.",
            "days": days,
        }
        return state["weekly_schedule"]

    # Track how much time each category has already received this week
    assigned_minutes_by_category = {k: 0 for k in target_minutes_by_category.keys()}

    # Track which activities have already been used from each bank
    used_activity_indices = {k: set() for k in target_minutes_by_category.keys()}

    day_names = list(days.keys())

    # Round-robin through days, always trying to fill without exceeding daily_time_min
    progress_made = True
    while progress_made:
        progress_made = False

        # Categories with the biggest remaining unmet need go first
        categories_in_priority_order = sorted(
            target_minutes_by_category.keys(),
            key=lambda k: target_minutes_by_category[k] - assigned_minutes_by_category[k],
            reverse=True,
        )

        for day_name in day_names:
            remaining_day_minutes = daily_time_min - days[day_name]["total_minutes"]
            if remaining_day_minutes <= 0:
                continue

            for category_key in categories_in_priority_order:
                remaining_cat_minutes = (
                    target_minutes_by_category[category_key] - assigned_minutes_by_category[category_key]
                )
                if remaining_cat_minutes <= 0:
                    continue

                bank = state["activity_banks"].get(category_key, {})
                activities = bank.get("activities", [])

                if not activities:
                    continue

                activity = pick_activity_that_fits(
                    activities=activities,
                    used_indices=used_activity_indices[category_key],
                    remaining_minutes=remaining_day_minutes,
                )

                if activity is None:
                    continue

                duration = int(activity.get("duration_min", 5))

                days[day_name]["items"].append({
                    "category_key": category_key,
                    "category": DOMAIN_CONFIG[category_key]["display"],
                    "title": activity.get("title"),
                    "instructions": activity.get("instructions"),
                    "duration_min": duration,
                    "materials": activity.get("materials", "common household items"),
                    "level": activity.get("level", "current_or_next"),
                })

                days[day_name]["total_minutes"] += duration
                assigned_minutes_by_category[category_key] += duration
                progress_made = True

                # move to next day after placing one activity
                break

    bonus_activity = pick_weekly_bonus_extended_activity(
    state,
    extended_in_schedule_threshold=15,
    bonus_extended_cap_min=20,
    )

    schedule = {
        "status": "success",
        "summary": "Weekly schedule built from category activity banks and true daily minute limits.",
        "daily_time_min": daily_time_min,
        "target_minutes_by_category": target_minutes_by_category,
        "assigned_minutes_by_category": assigned_minutes_by_category,
        "days": days,
        "weekly_bonus_activity": bonus_activity,
    }

    state["weekly_schedule"] = schedule
    return schedule

In [19]:
# -------------------------------
# Case summary 
# -------------------------------
def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []

    allocation = state.get("weekly_slot_allocation", {}).get("allocation_by_category", {})

    for category_key in DOMAIN_CONFIG:
        delay_info = state["delay_estimates"].get(category_key, {})
        dev_age = state["dev_age"].get(category_key)

        chrono_for_gap = min(child.get("chronological_months", 0), 60)
        milestone_gap = None if dev_age is None else max(0, chrono_for_gap - dev_age)

        bank = state.get("activity_banks", {}).get(category_key, {})

        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "concern": child.get("concern"),
            "daily_time_min": child.get("daily_time_min"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": delay_info.get("delay_months"),
            "delay_reason": delay_info.get("reason", ""),
            "estimated_dev_age_months": dev_age,
            "milestone_gap_months": milestone_gap,
            "needs_special_support": not no_special_support_needed(state, category_key),
            "weekly_slots_for_category": allocation.get(category_key, 0),
            "activity_bank_status": bank.get("status", "missing"),
            "activity_bank_summary": bank.get("summary", ""),
        })

    return pd.DataFrame(rows)

In [20]:
# -------------------------------
# display_weekly_schedule(
# --------------------------------
def display_weekly_schedule(state: Dict[str, Any]) -> None:
    schedule = state.get("weekly_schedule", {})

    print_banner("WEEKLY SCHEDULE")

    if schedule.get("status") == "no_special_support":
        print(schedule.get("summary", "No weekly schedule available."))
        return

    print("Summary:", schedule.get("summary", ""))
    print("Daily time budget:", schedule.get("daily_time_min"))
    print("Target minutes by category:", schedule.get("target_minutes_by_category", {}))
    print("Assigned minutes by category:", schedule.get("assigned_minutes_by_category", {}))

    for day_name, day_info in schedule.get("days", {}).items():
        print("\n" + "-" * 100)
        print(f"{day_name.upper()} — total {day_info.get('total_minutes', 0)} min")
        print("-" * 100)

        items = day_info.get("items", [])
        if not items:
            print("No scheduled activities.")
            continue

        for item in items:
            print(f"- [{item['category']}] {item['title']} ({item['duration_min']} min)")
            print(f"  Instructions: {item['instructions']}")
            print(f"  Materials: {item['materials']}")

    bonus = schedule.get("weekly_bonus_activity")
    if bonus:
        print("\n" + "=" * 100)
        print("OPTIONAL WEEKLY BONUS ACTIVITY")
        print("=" * 100)
        print(f"- [{bonus['category']}] {bonus['title']} ({bonus['duration_min']} min)")
        print(f"  Instructions: {bonus['instructions']}")
        print(f"  Materials: {bonus['materials']}")
        if bonus.get("extended_reason"):
            print(f"  Why this can be longer: {bonus['extended_reason']}")
        print("  Note: This is optional and sits outside the normal daily time budget.")

In [21]:
# ----------------------------------
# Helper function to print profile 
# ----------------------------------

def print_child_reference(state: Dict[str, Any]) -> None:
    child = state["child"]

    print("\n" + "=" * 100)
    print("CHILD REFERENCE CHECK")
    print("=" * 100)
    print(f"Name: {child.get('name', '')}")
    print(f"Chronological age: {child.get('chronological_months', '')} months")
    print(f"Diagnosis / condition: {child.get('diagnosis', '')}")
    print(f"Parent concern: {child.get('concern', '')}")
    print(f"Daily time available: {child.get('daily_time_min', '')} minutes")

In [22]:
# -------------------------
# Run all - live 
# -------------------------
def run_all_categories_live():
    state = new_state()
    intake_agent_live(state)

    # Print child info here, right after intake and before delay estimation
    print_child_reference(state)

    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(
            state,
            category_key,
            min_yes_confirm=2,
            yes_ratio_confirm=0.70,
        )

    # Build category-level activity banks
    for category_key in DOMAIN_CONFIG:
        generate_category_activity_bank(state, category_key)

    # Allocate weekly slots and build actual weekly plan
    allocate_weekly_slots(state)
    build_weekly_schedule(state)

    summary_df = summarizer_agent(state)
    display_weekly_schedule(state)

    return state, summary_df

In [23]:
# Quick sanity check 
display(cdc_df.head())

,months,category,milestone,category_key,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_and_emotional,social_and_emotional_2_1
1,2,social and emotial,looks at your face,social_and_emotional,social_and_emotional_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_and_emotional,social_and_emotional_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_and_emotional,social_and_emotional_2_4
4,2,language and communication,makes sounds other than crying,language_and_communication,language_and_communication_2_5


In [25]:
# Run fa full live session across all 4 cathegories , age 0-5
state, summary_df = run_all_categories_live()
summary_df


INTAKE AGENT

CHILD REFERENCE CHECK
Name: jack
Chronological age: 51 months
Diagnosis / condition: none
Parent concern: overal developmental delay specially speech
Daily time available: 10 minutes

DELAY ESTIMATOR AGENT
Movement / Physical: 0 month starting delay | There is little reason to think this domain is significantly affected.
Social / Emotional: 6 month starting delay | This domain may be mildly affected but not severely delayed.
Language / Communication: 12 month starting delay | Language development is likely delayed based on the parent's concern.
Cognitive / Adaptive: 6 month starting delay | Cognitive and adaptive development may be affected due to overall developmental delay concerns.

MILESTONE Q&A AGENT — Movement / Physical

--- Asking Movement / Physical milestones around 48 months ---

--- Asking Movement / Physical milestones around 60 months ---

Question / answer transcript:
- [48m] catches a large ball most of the times -> yes
- [48m] holds crayon or pencil betw

,child_name,chronological_age_months,diagnosis,concern,daily_time_min,category,estimated_delay_months,delay_reason,estimated_dev_age_months,milestone_gap_months,needs_special_support,weekly_slots_for_category,activity_bank_status,activity_bank_summary
0,jack,51,none,overal developmental delay specially speech,10,Movement / Physical,0,There is little reason to think this domain is...,48,3,False,0,no_special_support,We do not think jack has a meaningful delay in...
1,jack,51,none,overal developmental delay specially speech,10,Social / Emotional,6,This domain may be mildly affected but not sev...,48,3,False,0,no_special_support,We do not think jack has a meaningful delay in...
2,jack,51,none,overal developmental delay specially speech,10,Language / Communication,12,Language development is likely delayed based o...,30,21,True,0,success,This activity bank includes engaging and pract...
3,jack,51,none,overal developmental delay specially speech,10,Cognitive / Adaptive,6,Cognitive and adaptive development may be affe...,36,15,True,0,success,This activity bank includes engaging and pract...


In [ ]:
# ---------------------------------------
# Run 10 test cases 
# ---------------------------------------

cases = [
    {"case_id": "C01", "child_name": "Noah", "age_months": 10, "age_label": "10 months",
     "diagnosis": "Down syndrome",
     "primary_concern": "Not sitting independently yet",
     "secondary_features": "Hypotonia, slow feeding, socially engaged",
     "review_focus": "infant motor + early planning"},

    {"case_id": "C02", "child_name": "Maya", "age_months": 18, "age_label": "18 months",
     "diagnosis": "No formal diagnosis",
     "primary_concern": "No words yet",
     "secondary_features": "Limited pointing, frustration communicating",
     "review_focus": "early language delay"},

    {"case_id": "C03", "child_name": "Leo", "age_months": 26, "age_label": "2y 2m",
     "diagnosis": "Autism spectrum disorder",
     "primary_concern": "Limited eye contact and few words",
     "secondary_features": "Repetitive play, transitions hard, picky eating",
     "review_focus": "social communication + behavior"},

    {"case_id": "C04", "child_name": "Sofia", "age_months": 37, "age_label": "3y 1m",
     "diagnosis": "Prematurity history",
     "primary_concern": "Speech hard to understand",
     "secondary_features": "Fine motor delay, good comprehension",
     "review_focus": "speech intelligibility + fine motor"},

    {"case_id": "C05", "child_name": "Ethan", "age_months": 54, "age_label": "4y 6m",
     "diagnosis": "Cerebral palsy",
     "primary_concern": "Trouble keeping up in preschool",
     "secondary_features": "Self-care delays, frequent falls",
     "review_focus": "motor + adaptive function"},

    {"case_id": "C06", "child_name": "Amina", "age_months": 60, "age_label": "5 years",
     "diagnosis": "Global developmental delay",
     "primary_concern": "Learning and routines are hard",
     "secondary_features": "Short attention span, follows only simple directions",
     "review_focus": "broad developmental support"},

    {"case_id": "C07", "child_name": "Lucas", "age_months": 14, "age_label": "14 months",
     "diagnosis": "Fragile X syndrome",
     "primary_concern": "Not walking, limited babbling",
     "secondary_features": "Sensory sensitivity, shy socially",
     "review_focus": "mixed motor/language/sensory"},

    {"case_id": "C08", "child_name": "Emma", "age_months": 32, "age_label": "2y 8m",
     "diagnosis": "Suspected childhood apraxia of speech",
     "primary_concern": "Very limited speech output",
     "secondary_features": "Uses gestures, gets frustrated, inconsistent sounds",
     "review_focus": "speech disorder specificity"},

    {"case_id": "C09", "child_name": "Zain", "age_months": 48, "age_label": "4 years",
     "diagnosis": "ADHD / attention-regulation concerns",
     "primary_concern": "Very active, impulsive, and struggles to stay with structured activities",
     "secondary_features": "Big emotional reactions, interrupts, difficulty following routines, social interest intact",
     "review_focus": "attention + regulation + functional behavior"},

    {"case_id": "C10", "child_name": "Isla", "age_months": 44, "age_label": "3y 8m",
     "diagnosis": "SYNGAP1-related disorder",
     "primary_concern": "Delayed speech and balance",
     "secondary_features": "Attention issues, clumsy gait, limited pretend play",
     "review_focus": "rare condition usefulness"},
]

df_cases = pd.DataFrame(cases)
print(df_cases)